# 01 - Exploration du fichier PGN Lichess
Objectif : comprendre la structure du fichier sur 10k parties, puis les sauver en Parquet.
Le code de lecture est dans `pgn.py`.

In [1]:
import itertools
from pathlib import Path

from pgn import open_pgn, read_games

PGN_PATH = Path("../Data/lichess_db_standard_rated_2026-09.pgn.zst")
PARQUET_PATH = Path("../Data/sample_10k.parquet")

## 1. Le fichier brut
Ce n'est pas un tableau mais du texte (format PGN). Chaque partie =
- des lignes d'en-têtes `[Clé "valeur"]` (joueurs, Elo, résultat, cadence...)
- une ligne vide
- **une seule ligne** avec tous les coups : `1. e4 { [%clk 0:02:00] } 1... c5 { [%clk 0:02:00] } ...`

`1.` = coup des blancs, `1...` = coup des noirs, `{ [%clk ...] }` = temps restant à la pendule.

In [8]:
with open_pgn(PGN_PATH) as f:
    for line in itertools.islice(f, 400):
        print(line[:5000], end="")

[Event "Rated Bullet game"]
[Site "https://lichess.org/bcRJNu4b"]
[Date "2026.09.01"]
[Round "-"]
[White "Mycoba"]
[Black "Sentenced222"]
[Result "0-1"]
[UTCDate "2026.09.01"]
[UTCTime "00:00:20"]
[WhiteElo "1058"]
[BlackElo "1029"]
[WhiteRatingDiff "-7"]
[BlackRatingDiff "+11"]
[ECO "B21"]
[Opening "Sicilian Defense: Smith-Morra Gambit"]
[TimeControl "120+1"]
[Termination "Normal"]

1. e4 { [%clk 0:02:00] } 1... c5 { [%clk 0:02:00] } 2. d4 { [%clk 0:02:00] } 2... cxd4 { [%clk 0:01:59] } 3. Qxd4 { [%clk 0:02:01] } 3... Nc6 { [%clk 0:01:59] } 4. Qd1 { [%clk 0:02:01] } 4... Nf6 { [%clk 0:01:59] } 5. Nc3 { [%clk 0:02:01] } 5... d6 { [%clk 0:01:58] } 6. Nf3 { [%clk 0:01:59] } 6... Bg4 { [%clk 0:01:57] } 7. e5 { [%clk 0:01:58] } 7... dxe5 { [%clk 0:01:51] } 8. Qxd8+ { [%clk 0:01:57] } 8... Rxd8 { [%clk 0:01:52] } 9. Bb5 { [%clk 0:01:53] } 9... a6 { [%clk 0:01:45] } 10. Bxc6+ { [%clk 0:01:53] } 10... bxc6 { [%clk 0:01:46] } 11. Nxe5 { [%clk 0:01:51] } 11... e6 { [%clk 0:01:36] } 12. Nxc6 { [

## 2. Lecture de 10k parties -> une ligne par partie
- chaque en-tête devient une colonne, la ligne de coups brute va dans `Movetext`
- `moves` : liste des coups (sans commentaires `{...}`, numéros `1.` / `1...` ni résultat final)

In [12]:
df = read_games(PGN_PATH, n_read=10_000)
df.write_parquet(PARQUET_PATH)
print(df.shape)
df.select("WhiteElo", "BlackElo", "Event", "Result", "moves", "n_moves").head()

(10000, 22)


WhiteElo,BlackElo,Event,Result,moves,n_moves
i32,i32,str,str,list[str],u32
1058,1029,"""Rated Bullet game""","""0-1""","[""e4"", ""c5"", … ""Rf1#""]",86
896,943,"""Rated Bullet game""","""1-0""","[""e4"", ""e5"", … ""Nf6+""]",39
1542,1554,"""Rated Bullet game""","""0-1""","[""d4"", ""e6"", … ""Rxb8""]",44
852,880,"""Rated Bullet game""","""0-1""","[""e4"", ""e5"", … ""Qh6""]",42
1149,1216,"""Rated Bullet game""","""0-1""","[""Nf3"", ""Nf6"", … ""Qxb2""]",44


## 3. Exploration rapide

In [13]:
df.schema

Schema([('Event', String),
        ('Site', String),
        ('Date', String),
        ('Round', String),
        ('White', String),
        ('Black', String),
        ('Result', String),
        ('UTCDate', String),
        ('UTCTime', String),
        ('WhiteElo', Int32),
        ('BlackElo', Int32),
        ('WhiteRatingDiff', Int32),
        ('BlackRatingDiff', Int32),
        ('ECO', String),
        ('Opening', String),
        ('TimeControl', String),
        ('Termination', String),
        ('Movetext', String),
        ('WhiteTitle', String),
        ('BlackTitle', String),
        ('moves', List(String)),
        ('n_moves', UInt32)])

In [14]:
# Event contient parfois une URL de tournoi -> on garde juste la cadence
df["Event"].str.extract(r"(UltraBullet|Bullet|Blitz|Rapid|Classical|Correspondence)").value_counts(sort=True)

Event,count
str,u32
"""Blitz""",4654
"""Bullet""",3568
"""Rapid""",1641
"""UltraBullet""",75
"""Classical""",57
"""Correspondence""",5


In [15]:
df["Result"].value_counts(sort=True)

Result,count
str,u32
"""1-0""",4935
"""0-1""",4709
"""1/2-1/2""",355
"""*""",1


In [16]:
df.select("WhiteElo", "BlackElo", "n_moves").describe()

statistic,WhiteElo,BlackElo,n_moves
str,f64,f64,f64
"""count""",10000.0,10000.0,10000.0
"""null_count""",0.0,0.0,0.0
"""mean""",1640.3315,1639.4759,67.1244
"""std""",403.280535,402.427631,31.441787
"""min""",400.0,400.0,0.0
"""25%""",1358.0,1357.0,46.0
"""50%""",1651.0,1654.0,63.0
"""75%""",1924.0,1919.0,85.0
"""max""",3001.0,3026.0,553.0
